In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr                  
import cartopy.crs as ccrs           
import cartopy.feature as cfeature   

In [ ]:
#-------- Reading .nc data ----------
var = xr.open_dataset('../data/sst.mnmean.nc')   # Dataset : 파일 전체
sst = var['sst']                                  # DataArray : sst 변수 하나

print(type(var))
print(type(sst))

In [ ]:
print(sst)

In [ ]:
# sst는 (time, lat, lon) 순서의 3차원 자료
# 앞으로 나오는 선택(isel, sel)은 이 세 축 중 하나를 고정하는 일

# 순서(번호)로 고르기
# sst[0, :, :]                  # numpy 방식 — 첫 번째 시점
# sst.isel(time=0)              # 축 이름으로 — 첫 번째 시점 (같은 결과)

# 좌표 값으로 고르기
# sst.sel(time='1981-12-01')    # 1981년 12월
# sst.sel(lat=35, lon=130, method='nearest')   # 35N, 130E에 가장 가까운 격자

In [ ]:
grid = sst.sel(lat=35, lon=130, method='nearest')
print(grid.shape)

In [ ]:
plt.plot(grid['time'].values, grid)      
plt.title('SST at 35N, 130E')
plt.ylabel('SST (degC)')
plt.show()

In [ ]:
sst.isel(time=0).plot.contour()
# sst[0, :, :].plot.contour()
# sst.sel(time='1981-12-01').plot.contour()
plt.show()

In [ ]:
#------------- Plotting -------------
fig = plt.figure(figsize=(9, 7))
ax = plt.axes(projection=ccrs.PlateCarree())   # 이 칸은 지도라고 명시하는것 PlateCarree는 위도/경도 좌표계 

sst.isel(time=0).plot.contourf(ax=ax, levels=20, cmap='jet')  # inferno, plasma, viridis, cividis, coolwarm, RdBu_r, BrBG_r, PiYG_r, PRGn_r, Spectral_r 
ax.set_title('Sea Surface Temperature (SST)')
ax.coastlines() 
#ax.add_feature(ct.feature.GSHHSFeature(edgecolor='k')) 

plt.savefig('test.png', facecolor='w')   
plt.show()

In [ ]:
# projection= (axes 만들 때) : 그림을 어떤 지도로 그릴지 → 바꿔가며 씀
# transform=  (그릴 때)      : 내 자료가 어떤 좌표계인지 → 자료는 위경도이므로 항상 ccrs.PlateCarree()
# 3-2에서 transform이 없어도 됐던 건 projection도 PlateCarree라 우연히 맞았기 때문
# → Robinson으로 바꾸고 transform을 지워 보면 그림이 깨짐

In [ ]:
fig = plt.figure(figsize=(9, 7))
# ax = plt.axes(projection=ccrs.PlateCarree())               # 기본
# ax = plt.axes(projection=ccrs.Mercator())                  # Mercator
# ax = plt.axes(projection=ccrs.Mollweide())                 # Mollweide
# ax = plt.axes(projection=ccrs.Robinson())                  # Robinson
# ax = plt.axes(projection=ccrs.LambertConformal(130, 40))   # LambertConformal (중심 경도, 위도)
# ax = plt.axes(projection=ccrs.Orthographic())              # Orthographic (contourf가 깨지면 pcolormesh)

# ax.set_extent([90, 180, 0, 60], crs=ccrs.PlateCarree())    # 영역 선택 [서경, 동경, 남위, 북위]

sst.isel(time=0).plot.contourf(ax=ax,
                               levels=[-3, 0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30],  
                               cmap='jet',
                               transform=ccrs.PlateCarree(),        # 자료의 좌표계 — 항상 이것
                               cbar_kwargs={'extendrect': True, 'orientation': 'horizontal'})
ax.set_title('Sea Surface Temperature (SST)')
ax.coastlines()
plt.show()

In [ ]:
# fig = plt.figure()                 : 도화지
# plt.subplot(행열번호, projection=) : 도화지 위에 지도 칸 하나 (투영을 칸마다 다르게 줄 수 있음)
# .plot.contourf(ax=칸, transform=)  : 그 칸에 그리기 — transform은 자료 좌표계라 항상 PlateCarree
# ax.coastlines / gridlines / add_feature : 칸 꾸미기
#
# subplot 번호는 "행, 열, 몇 번째" 를 붙여 쓴 것. 1부터 셈 (subplots의 axes[0,0]과 다름)
#
#   221 = 2행 2열 중 1번        122 = 1행 2열 중 2번 → 오른쪽 전체
#   +-------+-------+           +-------+-------+
#   |  221  |       |           |       |       |
#   +-------+  122  |           |       |  122  |
#   |  223  |       |           |       |       |
#   +-------+-------+           +-------+-------+
#   (223 = 2행 2열 중 3번, 왼쪽 아래)
#=====================================


In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = plt.axes(projection=ccrs.Robinson(central_longitude=180)) # 중심 경도 180 → 태평양이 가운데

clevs = np.arange(-3, 33, 1)   # -3부터 32까지 1 간격 (np.linspace(-3, 33, 37)과 같음)

sst.isel(time=0).plot.contourf(ax=ax, levels=clevs, cmap='jet', transform=ccrs.PlateCarree(),
                               cbar_kwargs={'extendrect': True, 'orientation': 'horizontal'})
ax.set_title('Sea Surface Temperature (SST)')
ax.coastlines()
ax.gridlines(draw_labels=True, linewidth=1, linestyle=':', color='gray')   
ax.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '50m',    
                                            edgecolor='face', facecolor='black')) 
plt.show()

In [ ]:

fig = plt.figure(figsize=(18, 14), dpi=100)
clevs = np.arange(-3, 33, 1)


ax1 = plt.subplot(221, projection=ccrs.Mollweide(180))
sst.isel(time=0).plot.contourf(ax=ax1, levels=clevs, cmap='jet', transform=ccrs.PlateCarree(),
                               cbar_kwargs={'extendrect': True, 'orientation': 'horizontal', 'pad': 0.06, 'aspect': 30})
ax1.set_title('Sea Surface Temperature (SST)', fontsize=17)
ax1.coastlines()
ax1.gridlines(draw_labels=True, linewidth=1, linestyle=':', color='gray', x_inline=False)
ax1.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '50m', edgecolor='face', facecolor='black'))

ax2 = plt.subplot(223, projection=ccrs.NearsidePerspective(130, 40, satellite_height=1500000))
sst.isel(time=0).plot.pcolormesh(ax=ax2, levels=clevs, cmap='jet', transform=ccrs.PlateCarree(),
                                 cbar_kwargs={'extendrect': True, 'orientation': 'horizontal', 'pad': 0.06, 'fraction': 0.045})
ax2.set_title('Sea Surface Temperature (SST)', fontsize=17)
ax2.coastlines()
ax2.gridlines(draw_labels=True, linewidth=1, linestyle=':', color='gray')
ax2.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '10m', edgecolor='face', facecolor='black'))

ax3 = plt.subplot(122, projection=ccrs.Orthographic(130, 40))
sst.isel(time=0).plot.contourf(ax=ax3, levels=clevs, cmap='jet', transform=ccrs.PlateCarree(),
                               cbar_kwargs={'extendrect': True, 'orientation': 'vertical', 'shrink': 0.45})
ax3.set_title('Sea Surface Temperature (SST)', fontsize=17)
ax3.coastlines()
ax3.gridlines(draw_labels=True, linewidth=1, linestyle=':', color='gray')
ax3.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '50m', edgecolor='face', facecolor='black'))

plt.savefig('SST_map_on_different_projections.png')
plt.show()